# `bloqify` Adder

We show how to build a quantum-quantum adder by composing routines written with the `@bloqify` syntax.

In [ ]:
# Top-level qualtran imports
import qualtran as qlt
import qualtran.dtype as qdt

## `maj` majority computation and half-adder

In [ ]:
@qlt.bloqify
def maj(bb: 'BloqBuilder', ck, ik, tk):
    # Figure 2, left part: uses one AND
    ck, ik = bb.CNOT(ck, ik)
    ck, tk = bb.CNOT(ck, tk)
    [ik, tk], ckp1 = bb.And([ik, tk])
    ck, ckp1 = bb.CNOT(ck, ckp1)
    return {'ck': ck, 'ik': ik, 'tk': tk, 'ckp1': ckp1}

In [ ]:
bloq = maj.make(qlt.Signature.build(ck=1, ik=1, tk=1))
qlt.show_bloq(bloq, 'musical_score')

## `un_maj` inverse

In [ ]:
@qlt.bloqify
def un_maj(bb: 'BloqBuilder', ck, ik, tk, ckp1):
    ck, ckp1 = bb.CNOT(ck, ckp1)
    [ik, tk] = bb.UnAnd([ik, tk], ckp1)
    ck, ik = bb.CNOT(ck, ik)
    return {'ck': ck, 'ik': ik, 'tk': tk}

In [ ]:
bloq = un_maj.make(qlt.Signature.build(ck=1, ik=1, tk=1, ckp1=1))
qlt.show_bloq(bloq, 'musical_score')

## The adder

This `add_bits` function takes input qvars `i` and `t` as a vector of qubits. We'll wrap it later with casting from- and to integers.

In [ ]:
@qlt.bloqify
def add_bits(bb: 'BloqBuilder', i: 'QVarT', t: 'QVarT'):
    assert i.shape == t.shape
    assert len(i.shape) == 1, '1d array'
    n = i.shape[0]

    # First bit: no input carry
    [i[0], t[0]], c0 = bb.And([i[0], t[0]])
    c = [c0]

    # Ripple-carry
    for k in range(1, n - 1):
        c[k - 1], i[k], t[k], cnext = maj(bb, c[k - 1], i[k], t[k])
        c.append(cnext)

    # Last bit: no output carry
    k = n - 1
    c[k - 1], t[k] = bb.CNOT(c[k - 1], t[k])

    # Un-ripple-carry
    for k in range(n - 1 - 1, 1 - 1, -1):
        c[k - 1], i[k], t[k] = un_maj(bb, ck=c[k - 1], ik=i[k], tk=t[k], ckp1=c[k])

    # First bit
    i[0], t[0] = bb.UnAnd([i[0], t[0]], c[0])

    for k in range(n):
        i[k], t[k] = bb.CNOT(i[k], t[k])

    return {'i': i, 't': t}

In [ ]:
bloq = add_bits.make(qlt.qsig(i=qdt.QBit()[4], t=qdt.QBit()[4]))
qlt.show_bloq(bloq)

### QLT IR

In [ ]:
from qualtran.l1 import dump_l1
print(dump_l1(bloq))

## The top-level `add` function

In [ ]:
@qlt.bloqify
def add(bb: 'BloqBuilder', a: 'QVar', b: 'QVar'):
    assert a.dtype.num_bits == b.dtype.num_bits

    i = a[:]  # split
    t = b[:]  # split

    # Account for endianness!
    i_out, t_out = add_bits.inline(bb, i=i[::-1], t=t[::-1])
    return {'a': bb.join(i_out[::-1], dtype=a.dtype), 'b': bb.join(t_out[::-1], dtype=b.dtype)}

In [ ]:
n = 4
bloq = add.make(qlt.qsig(a=qdt.QUInt(n), b=qdt.QUInt(n)))

### Classical check

In [ ]:
for a in range(2**n):
    for b in range(2**n):
        a_out, apb = bloq.call_classically(a=a, b=b)
        print(f'{a:5d} {b:5d} -> {apb:5d}  ({a+b:5d}) {"*" if apb != (a+b)%2**n else ""}')